# 03 - Hard-Negative Generation (item 4)

Runs all six strategies from `hard_negatives.py`. Read that module's docstring before using `topically_similar_swap` / `same_author_swap` outputs as training negatives -- they are not guaranteed-false the way the other four strategies are.

In [ ]:
# == Environment + paths =========================================================
import json, os, re, sys, time
from pathlib import Path

# Kaggle detection uses an env var Kaggle actually sets, not path existence: on
# Windows, Path('/kaggle/working').exists() is NOT a reliable off-Kaggle check --
# pathlib treats a leading '/' as relative to the current drive's root, so an
# unrelated local C:\kaggle or F:\kaggle directory would silently return True.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"))

def _find(name, prefer_largest=True):
    """Search for `name`; if multiple matches exist, prefer the largest file --
    a complete/authoritative cache is virtually always bigger than a stale or
    partial one left over from an earlier run."""
    roots = ["/kaggle/input"] if ON_KAGGLE else ["..", "../..", "."]
    hits = []
    for root in roots:
        r = Path(root)
        if r.exists():
            hits.extend(r.rglob(name))
    if not hits:
        return None
    if prefer_largest:
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
    return hits[0]

def _find_src_dir():
    """Locate the directory holding this project's shared modules (identified by
    containing graph_utils.py, not by a fixed relative path) -- on Kaggle, an
    attached dataset lands under /kaggle/input/<dataset-name>/..., so neither
    '../src' nor 'src' exists there; search by content the same way _find() above
    locates data files, instead of assuming a fixed layout."""
    for c in (Path("../src"), Path("src"), Path("/kaggle/working/src")):
        if (c / "graph_utils.py").exists():
            return c
    if ON_KAGGLE:
        hits = list(Path("/kaggle/input").rglob("graph_utils.py"))
        if hits:
            return hits[0].parent
    return Path("src")  # last resort; will raise ModuleNotFoundError on import if wrong

SRC_DIR = _find_src_dir()
sys.path.insert(0, str(SRC_DIR))

BENCH = _find("scireqa_full_relational_v1.1_humanmerged.jsonl") or _find("scireqa_full_relational_v1.1.jsonl")
REFS_CACHE = _find("citecheck_refs_cache.json")
CITE_RESULTS = _find("citecheck_results.json")
BENCH_VERIFIED = _find("scireqa_human_verified_subset.jsonl")
assert BENCH and REFS_CACHE and CITE_RESULTS, "attach the SciRecBench + CiteCheck output dataset"

OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE:", ON_KAGGLE)
print("SRC_DIR:", SRC_DIR.resolve() if SRC_DIR.exists() else f"{SRC_DIR} (NOT FOUND -- attach the src/ dataset)")
print("BENCH:", BENCH, f"({BENCH.stat().st_size:,} bytes)" if BENCH else "")
print("REFS_CACHE:", REFS_CACHE, f"({REFS_CACHE.stat().st_size:,} bytes)" if REFS_CACHE else "")
print("CITE_RESULTS:", CITE_RESULTS)
print("BENCH_VERIFIED:", BENCH_VERIFIED)
print("OUT_DIR:", OUT_DIR)


In [ ]:
import random, json
from collections import defaultdict
from dataclasses import asdict
from graph_utils import CitationGraph, load_relational_claims
from hard_negatives import (gen_direction_reversal, gen_relation_substitution,
    gen_shared_refs_no_dependency, gen_temporally_impossible, gen_topically_similar_swap,
    gen_same_author_swap, build_tfidf_index, load_arxiv_authors, summarize)

graph = CitationGraph.load(REFS_CACHE)
claims = load_relational_claims(BENCH, CITE_RESULTS)
claims = [c for c in claims if c["recorded_verdict"] is not None]
print(f"Loaded {len(claims)} relational claims, graph has {len(graph)} nodes.")

rng = random.Random(0)
all_negatives = []
all_negatives += gen_direction_reversal(claims, graph, rng)
all_negatives += gen_relation_substitution(claims, graph, rng)
all_negatives += gen_shared_refs_no_dependency(claims, graph, rng)
all_negatives += gen_temporally_impossible(claims, graph, rng)

print("Building TF-IDF index over benchmark abstracts...")
paper_abstracts, candidates_by_domain = {}, defaultdict(set)
with open(BENCH, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        dom = rec.get("domain")
        for gt in rec.get("ground_truth_papers") or []:
            pid, ab = gt.get("paper_id"), gt.get("abstract")
            if pid and ab:
                paper_abstracts[pid] = ab
                candidates_by_domain[dom].add(pid)
tfidf_vectors = build_tfidf_index(paper_abstracts)
candidates_by_domain = {k: list(v) for k, v in candidates_by_domain.items()}
print(f"  {len(paper_abstracts)} papers with abstracts")
all_negatives += gen_topically_similar_swap(claims, graph, rng, paper_abstracts,
                                             tfidf_vectors, candidates_by_domain)

arxiv_meta = _find("papers_metadata_10k_backup.csv")
if arxiv_meta:
    authors_by_paper = load_arxiv_authors(arxiv_meta)
    print(f"  {len(authors_by_paper)} arXiv papers with author lists")
    all_negatives += gen_same_author_swap(claims, graph, rng, authors_by_paper)
else:
    print("  papers_metadata_10k_backup.csv not found -- skipping same_author_swap")

print(f"\nGenerated {len(all_negatives)} hard negatives.\n")
summary = summarize(all_negatives)
print(f"{'strategy':26s} {'n':>7s} {'caught':>8s} {'fooled rate':>12s}")
for strat, s in summary.items():
    print(f"{strat:26s} {s['n']:7d} {s['caught_by_current_verifier']:8d} {s['fooled_rate']:12.1%}")

with open(OUT_DIR / "hard_negatives.jsonl", "w", encoding="utf-8") as f:
    for n in all_negatives:
        f.write(json.dumps(asdict(n)) + "\n")
with open(OUT_DIR / "hard_negatives_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved to {OUT_DIR}")
